# Paddy YOLO EMA DySample RGG Fold0

**300 epochs**, nano scale, 640 pixels, total batch 32 across two GPUs (16 each),
`yolov8n.pt` initialization.
AdamW uses `lr0=0.001` and `warmup_bias_lr=0.0` for this corrected RGG rerun.
RGG convenience aliases no longer register duplicate state tensors during weight averaging.
Three EMA blocks follow the P3/P4/P5 RGG blocks; two DySample layers replace FPN upsampling.
Bounding-box training uses DIMIoU H, with BCE classification and DFL.
Dataset: `dmrashidpferrer/r2000-pestnet`. This single experiment preserves its supplied
train/validation/test split; `fold0` is a run label, not a regenerated cross-validation fold.
Compatible pretrained tensors are transferred by corresponding layer, accounting for index shifts.
New EMA/DySample parameters and incompatible COCO classifier tensors are initialized separately.
GPU and Internet are enabled in the accompanying private notebook metadata.

In [ ]:
import base64, io, os, sys, zipfile
from pathlib import Path
PROJECT = Path('/kaggle/working/paddy_yolo_ema_dysample_source')
PROJECT.mkdir(parents=True, exist_ok=True)
B64_SOURCE = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(B64_SOURCE))) as archive:
    archive.extractall(PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
os.environ['YOLO_CONFIG_DIR'] = '/kaggle/working/ema_dysample_settings/ultralytics'
os.environ['MPLCONFIGDIR'] = '/kaggle/working/ema_dysample_settings/matplotlib'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['PYTHONUNBUFFERED'] = '1'
print('Extracted project:', PROJECT)

In [ ]:
import json, subprocess
RUN = json.loads((PROJECT / 'kaggle/ema_dysample/run_config.json').read_text())
print(json.dumps(RUN, indent=2))
sys.path.insert(0, str(PROJECT / 'kaggle/ema_dysample'))
from runtime import configure_ultralytics, run_logged
run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r',
            str(PROJECT / 'kaggle/ema_dysample/requirements.txt')],
           Path('/kaggle/working/environment_setup.log'), PROJECT)
import torch, torchvision, ultralytics
assert torch.cuda.is_available(), 'This run requires a Kaggle GPU.'
assert ultralytics.__version__ == '8.3.18'
configure_ultralytics()
print('Torch:', torch.__version__, 'Torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
run_logged([sys.executable, '-m', 'pytest', '-q', '--tb=short'],
           Path('/kaggle/working/preflight_tests.log'), PROJECT)

In [ ]:
sys.path.insert(0, str(PROJECT / 'kaggle/baseline'))
from prepare_dataset import prepare_dataset
DATA = prepare_dataset(Path('/kaggle/input'), Path('/kaggle/working/ema_dysample_data'))
# Download official COCO YOLOv8-N weights; no Kaggle credentials are needed here.
from ultralytics import YOLO
pretrained = YOLO(RUN['initial_weights'])
print('Initial weights ready:', RUN['initial_weights'])
del pretrained

In [ ]:
OUTPUT = Path('/kaggle/working/ema_dysample_output')
command = [sys.executable, str(PROJECT / 'kaggle/ema_dysample/run_training.py'),
           '--data', str(DATA), '--output-dir', str(OUTPUT),
           '--epochs', str(RUN['epochs']), '--batch', str(RUN['batch']),
           '--imgsz', str(RUN['imgsz']), '--workers', str(RUN['workers']),
           '--device', RUN['device'], '--weights', RUN['initial_weights'],
           '--hyp', str(PROJECT / RUN['hyp']), '--name', RUN['run_name'],
           '--patience', str(RUN['patience']), '--save-period', str(RUN['save_period'])]
run_logged(command, OUTPUT / 'training.log', PROJECT)

In [ ]:
print((OUTPUT / 'metrics.json').read_text())
print((OUTPUT / 'weight_transfer.json').read_text())
print('Download ema_dysample_output for checkpoints, results.csv and validation/test plots.')
print('Dataset audit:', DATA.parent / 'dataset_audit.json')